# 1. Simple XMCD: phase retrieval only

Run the cells in order. This demonstration uses synthetic intensities; iteration counts are illustrative, not convergence guarantees. Replace the data-generation cell with your normalized experimental intensities, support and detector mask. See [the examples guide](../docs/universal_reconstruction_examples.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "library" / "phase_retrieval_universal.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from library import phase_retrieval_universal as universal

# Small synthetic data make this notebook runnable without experimental files.
shape = (32, 32)
y, x = np.indices(shape)
supportmask = ((x - 16)**2 + (y - 16)**2 < 7**2).astype(float)
mask_pixel = np.zeros(shape, dtype=int)  # 0 measured; 1 invalid/unmeasured.
magnetic_pattern = np.tanh((x - 16) / 2) * supportmask

def to_field(obj):
    return np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(obj)))

def to_object(field):
    return np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(field)))


In [ ]:
obj_pos_true = supportmask * np.exp(-0.2 + 0.2j + (0.04 + 0.02j) * magnetic_pattern)
obj_neg_true = supportmask * np.exp(-0.2 + 0.2j - (0.04 + 0.02j) * magnetic_pattern)
holograms = np.stack([abs(to_field(obj_pos_true))**2, abs(to_field(obj_neg_true))**2])
# Experimental replacement: np.stack([pos_intensity, neg_intensity]).


In [ ]:
recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    modes=[1], mode_initialization="support_fft",
    warmup_mode=["HAPRE", "ER"], warmup_Nit=[40, 10],
    inner_mode=["ER"], inner_Nit=[10], outer_iterations=3,
    shuffle_observations=False,
    projection_relaxation=0.3,
    final_projection_relaxation=0,
    random_seed=0,
)
recipe.update(projection_model="none", partial_coherence=False,
              constrain_nonphysical_modes_common=False)
universal.print_universal_workflow(recipe, state_labels=["sample", "sample"])
fields, warmup, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, supportmask,
    state_labels=["sample", "sample"], energy_labels=[780., 780.],
    polarization_coefficients=[+1, -1], illumination_labels=["beam", "beam"],
    universal_recipe=recipe,
)
obj_pos, obj_neg = [to_object(f) for f in fields]


`none` disables joint physical/spectral fitting, while detector and support constraints remain active. For experimental XMCD, align the two objects and match their complex scale in a nonmagnetic region before interpreting the log ratio. Its phase is wrapped; it is not a calibrated magnetization map.

In [ ]:
floor = 1e-6 * max(abs(obj_pos).max(), abs(obj_neg).max())
valid = (supportmask != 0) & (abs(obj_pos) > floor) & (abs(obj_neg) > floor)
xmcd = np.full(shape, np.nan + 1j*np.nan)
xmcd[valid] = np.log(obj_pos[valid] / obj_neg[valid])
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(xmcd.real, cmap="RdBu_r"); axes[0].set_title("Log-amplitude contrast")
axes[1].imshow(xmcd.imag, cmap="RdBu_r"); axes[1].set_title("Wrapped phase contrast")
plt.tight_layout(); plt.show()
